In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

BASE_DIR = Path("__file__").resolve().parent.parent
RAW_DIR  = BASE_DIR / "data" / "raw"


# read raw files
crm     = pd.read_csv(RAW_DIR / "crm_customers.csv")
billing = pd.read_csv(RAW_DIR / "billing_transactions.csv")
churn   = pd.read_csv(RAW_DIR / "churn_data.csv")

In [2]:
# original data shapes

print(f"CRM shape:     {crm.shape}")
print(f"Billing shape: {billing.shape}")
print(f"Churn shape:   {churn.shape}")

CRM shape:     (95, 8)
Billing shape: (621, 6)
Churn shape:   (18, 3)


In [3]:
# 1. Inspect CRM Data

crm.head(10)

,customer_id,company_name,industry,city,plan,pipeline_stage,signup_date,account_manager
0,CUST-001,Company 1,Construction,Toronto,Starter,closed_won,23/11/2024,AM-4
1,CUST-002,Company 2,Retail,Calgary,Starter,NaN,2024-02-27,AM-5
2,CUST-003,Company 3,Education,Vancouver,Starter,Closed Won,2024-01-13,AM-4
3,CUST-004,Company 4,Education,Edmonton,Starter,Demo,2024-05-20,AM-2
4,CUST-005,Company 5,Education,Ottawa,Enterprise,Qualified,2024-05-05,AM-5
5,CUST-006,Company 6,Finance,Ottawa,Growth,Demo,2024-04-24,AM-2
6,CUST-007,Company 7,Logistics,Edmonton,Starter,Closed Lost,2024-03-12,AM-1
7,CUST-008,Company 8,Healthcare,Montreal,Starter,closed_won,2024-02-22,AM-4
8,CUST-009,Company 9,Logistics,Calgary,Growth,Demo,2024-12-12,AM-4
9,CUST-010,Company 10,Logistics,Vancouver,Enterprise,Won,2024-10-06,AM-1


In [4]:
crm.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   customer_id      95 non-null     str  
 1   company_name     95 non-null     str  
 2   industry         95 non-null     str  
 3   city             95 non-null     str  
 4   plan             95 non-null     str  
 5   pipeline_stage   88 non-null     str  
 6   signup_date      95 non-null     str  
 7   account_manager  95 non-null     str  
dtypes: str(8)
memory usage: 11.9 KB


In [5]:
# Only pipeline_stage column has nulls. These will be replaced as `Lead` in the silver layer

null_stages = crm[crm["pipeline_stage"].isna()]
print(f"Rows with null pipeline_stage: {len(null_stages)}")
null_stages.head(10)


Rows with null pipeline_stage: 7


,customer_id,company_name,industry,city,plan,pipeline_stage,signup_date,account_manager
1,CUST-002,Company 2,Retail,Calgary,Starter,NaN,2024-02-27,AM-5
38,CUST-039,Company 39,Finance,Toronto,Growth,NaN,2024-04-20,AM-2
53,CUST-054,Company 54,Retail,Toronto,Growth,NaN,2024-02-10,AM-2
57,CUST-058,Company 58,Healthcare,Vancouver,Starter,NaN,2024-11-12,AM-5
66,CUST-067,Company 67,Retail,Vancouver,Starter,NaN,2024-05-28,AM-2
69,CUST-070,Company 70,Healthcare,Winnipeg,Starter,NaN,2024-02-21,AM-1
77,CUST-078,Company 78,Retail,Calgary,Starter,NaN,2024-06-30,AM-3


In [6]:
# Inspect unique values for selected columns 

selected_columns = ["industry", "plan", "pipeline_stage"]

print("All unique values for selected columns:\n")

for col in selected_columns:
    print(crm[col].value_counts(dropna=False), "\n")


All unique values for selected columns:

industry
Education       21
Logistics       21
Retail          16
Finance         15
Healthcare      15
Construction     7
Name: count, dtype: int64 

plan
Starter       43
Enterprise    31
Growth        21
Name: count, dtype: int64 

pipeline_stage
Closed Lost    17
Lead           17
Proposal       15
Demo            8
Qualified       8
NaN             7
Closed Won      7
Won             6
closed_won      4
proposal        3
QUALIFIED       3
Name: count, dtype: int64 



In [7]:
# Flag the non-standard values in pipeline_stage to be fixed in the silver layer
standard_stages = {"Lead", "Qualified", "Demo", "Proposal", "Closed Won", "Closed Lost"}
non_standard = crm[~crm["pipeline_stage"].isin(standard_stages) & crm["pipeline_stage"].notna()]

print(f"Non-standard stage values: {len(non_standard)} rows\n")
print(non_standard["pipeline_stage"].value_counts())


Non-standard stage values: 16 rows

pipeline_stage
Won           6
closed_won    4
proposal      3
QUALIFIED     3
Name: count, dtype: int64


In [8]:
# Attempt standard parse.
# Rows that fail are the mixed-format ones
crm["date_parsed"] = pd.to_datetime(crm["signup_date"], format="%Y-%m-%d", errors="coerce")
mixed_format = crm[crm["date_parsed"].isna()]

# print results
print(f"Rows with non-standard date format: {len(mixed_format)}")
mixed_format[["customer_id", "signup_date"]].head(10)


Rows with non-standard date format: 10


,customer_id,signup_date
0,CUST-001,23/11/2024
10,CUST-011,14/02/2024
20,CUST-021,14/01/2024
30,CUST-031,22/05/2024
40,CUST-041,22/02/2024
50,CUST-051,01/10/2024
60,CUST-061,08/04/2024
70,CUST-071,13/07/2024
80,CUST-081,16/05/2024
90,CUST-091,24/03/2024


In [9]:
# Confirm all rows in crm data follow CUST-XXX pattern for customer_id

sample_ids = crm["customer_id"].head(5).tolist()
print("CRM customer_id sample:", sample_ids)

non_matching = crm[~crm["customer_id"].str.match(r"^CUST-\d+$")]
print(f"CRM IDs not matching CUST-XXX: {len(non_matching)}")


CRM customer_id sample: ['CUST-001', 'CUST-002', 'CUST-003', 'CUST-004', 'CUST-005']
CRM IDs not matching CUST-XXX: 0


In [10]:
# 2. Inspect Billing Data

billing.head(10)

,transaction_id,customer_id,amount,transaction_date,status,plan
0,TX-1-1,C001,149,2024-11-23,paid,Growth
1,TX-2-1,C002,499,2024-02-27,paid,Enterprise
2,TX-2-2,C002,499,2024-03-28,paid,Enterprise
3,TX-2-3,C002,499,2024-04-27,paid,Enterprise
4,TX-2-4,C002,499,2024-05-27,paid,Enterprise
5,TX-2-5,C002,499,2024-06-26,paid,Enterprise
6,TX-2-6,C002,499,2024-07-26,paid,Enterprise
7,TX-2-7,C002,499,2024-08-25,paid,Enterprise
8,TX-2-8,C002,499,2024-09-24,paid,Enterprise
9,TX-2-9,C002,499,2024-10-24,paid,Enterprise


In [11]:
billing.info()


<class 'pandas.DataFrame'>
RangeIndex: 621 entries, 0 to 620
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   transaction_id    621 non-null    str  
 1   customer_id       621 non-null    str  
 2   amount            621 non-null    int64
 3   transaction_date  621 non-null    str  
 4   status            621 non-null    str  
 5   plan              621 non-null    str  
dtypes: int64(1), str(5)
memory usage: 49.2 KB


In [12]:
# Inspect unique values for selected columns 

selected_columns = ["status", "plan"]

print("All unique values for selected columns:\n")

for col in selected_columns:
    print(billing[col].value_counts(dropna=False), "\n")


All unique values for selected columns:

status
paid        597
refunded     24
Name: count, dtype: int64 

plan
Growth        234
Enterprise    229
Starter       158
Name: count, dtype: int64 



In [13]:
# Confirm all rows in billing rm data follow CXXX pattern for customer_id
sample_billing_ids = billing["customer_id"].head(5).tolist()

print("Billing customer_id sample:", sample_billing_ids)

non_matching = billing[~billing["customer_id"].str.match(r"^C\d+$")]
print(f"Billing IDs not matching CXXX: {len(non_matching)}")

Billing customer_id sample: ['C001', 'C002', 'C002', 'C002', 'C002']
Billing IDs not matching CXXX: 0


In [14]:
# Show the difference side by side
print("\nCRM format:     CUST-001")
print("Billing format: C001")


CRM format:     CUST-001
Billing format: C001


### 2b. Duplicate transaction rows



In [15]:
duplicates = billing[billing.duplicated(subset="transaction_id", keep=False)]

print(f"Total rows involved in duplicates: {len(duplicates)}")
print(f"Unique transaction_ids that are duplicated: {duplicates['transaction_id'].nunique()}")
duplicates.sort_values("transaction_id").head(10)


Total rows involved in duplicates: 30
Unique transaction_ids that are duplicated: 15


,transaction_id,customer_id,amount,transaction_date,status,plan
85,TX-13-2,C013,499,2024-09-03,paid,Enterprise
86,TX-13-2,C013,499,2024-09-03,paid,Enterprise
106,TX-15-6,C015,49,2024-06-14,paid,Starter
107,TX-15-6,C015,49,2024-06-14,paid,Starter
143,TX-20-1,C020,149,2024-11-04,paid,Growth
144,TX-20-1,C020,149,2024-11-04,paid,Growth
167,TX-25-1,C025,499,2024-12-25,paid,Enterprise
168,TX-25-1,C025,499,2024-12-25,paid,Enterprise
224,TX-35-4,C035,149,2024-11-02,paid,Growth
225,TX-35-4,C035,149,2024-11-02,paid,Growth


### 2c. Refund rows mixed into billing

Refunds appear as negative-amount rows with `status = 'refunded'`. If we sum `amount` without filtering, revenue will be understated.

In [16]:
refunds = billing[billing["status"] == "refunded"]
refunds.head()

,transaction_id,customer_id,amount,transaction_date,status,plan
19,TX-3-9-R,C003,-149,2024-09-09,refunded,Growth
40,TX-6-2-R,C006,-149,2024-05-24,refunded,Growth
46,TX-6-7-R,C006,-149,2024-10-21,refunded,Growth
64,TX-8-7-R,C008,-499,2024-08-20,refunded,Enterprise
102,TX-15-3-R,C015,-49,2024-03-16,refunded,Starter


In [17]:
print(f"Refund rows: {len(refunds)}")
print(f"Total refund amount: ${refunds['amount'].sum():,.2f}")
print()

# Show what revenue looks like before vs after excluding refunds vs after excluding duplicates
gross_revenue   = billing["amount"].sum()
net_revenue_raw_dup = billing[billing["status"] == "paid"]["amount"].sum()

billing_clean = billing.drop_duplicates(subset="transaction_id", keep=False)
net_revenue_raw = billing_clean.loc[billing_clean["status"] == "paid", "amount"].sum()

print(f"Gross amount (incl. refunds):               ${gross_revenue:,.2f}")
print(f"Net amount (paid only, with duplicates):    ${net_revenue_raw_dup:,.2f}")
print(f"Net amount (paid only, without duplicates): ${net_revenue_raw:,.2f}")

Refund rows: 24
Total refund amount: $-6,076.00

Gross amount (incl. refunds):               $144,727.00
Net amount (paid only, with duplicates):    $150,803.00
Net amount (paid only, without duplicates): $142,033.00


In [18]:
# Standardize billing IDs temporarily for comparison
billing["customer_id_std"] = billing["customer_id"].str.replace("C", "CUST-", regex=False)
billing_only = set(billing["customer_id_std"].unique()) - set(crm["customer_id"].unique())
print(f"Customers in billing but not in CRM: {len(billing_only)}")
print(sorted(billing_only))

Customers in billing but not in CRM: 10
['CUST-096', 'CUST-097', 'CUST-098', 'CUST-099', 'CUST-100', 'CUST-101', 'CUST-102', 'CUST-103', 'CUST-104', 'CUST-105']


In [19]:
# 10 customer IDs appear in billing but have no matching record in the CRM. These are flagged in Silver with `billing_only_flag = True`.

In [20]:
# 3. Inspect Churn Data

churn.head(10)

,customer_id,churn_date,churn_reason
0,CUST-060,2024-07-25,Budget cuts
1,CUST-064,2024-03-05,Price
2,CUST-040,2024-10-07,Switched to competitor
3,CUST-003,2024-07-14,No longer needed
4,CUST-012,2024-09-23,No longer needed
5,CUST-051,2024-03-24,Poor support
6,CUST-065,2024-03-27,Budget cuts
7,CUST-059,2024-12-13,Switched to competitor
8,CUST-031,2024-09-02,Budget cuts
9,CUST-028,2024-07-26,Price


In [21]:
print(f"Churned customers: {len(churn)}")
print(f"\nChurn reasons:\n")
print(churn["churn_reason"].value_counts())

Churned customers: 18

Churn reasons:

churn_reason
Budget cuts               6
Switched to competitor    4
No longer needed          4
Price                     2
Poor support              2
Name: count, dtype: int64


In [22]:
# Confirm churn IDs match CRM format
sample_churn_ids = churn["customer_id"].head(5).tolist()
print("Churn customer_id sample:", sample_churn_ids)

non_matching_churn = churn[~churn["customer_id"].str.match(r"^CUST-\d+$")]
print(f"Churn IDs not matching CUST-XXX: {len(non_matching_churn)}")

Churn customer_id sample: ['CUST-060', 'CUST-064', 'CUST-040', 'CUST-003', 'CUST-012']
Churn IDs not matching CUST-XXX: 0
